## FOMC Event Study: Data Diagnosis and Construction

**Purpose:** This notebook diagnoses the reliability of the daily MGS yields, USDMYR and KLCI series, and reconstructs data for the FOMC event study. It checks how much of the 2015–2026 sample is jointly available inside the 1-day window before and after the FOMC meeting dates.

### 1. Setup and event window

- Load the FOMC meeting dates and shift each by +1 day. FOMC decisions are announced in the US afternoon, which is overnight in Malaysia, so the following day is the first Malaysian session that can react. This shifted date is `T+0`.
- Define the event window as `t-1`, `t+0`, `t+1` (`offset = (-1, 0, 1)`), giving 3 sessions per meeting.
- Build `df_bool`, a flag for MGS 10Y, KLCI and USDMYR that is `True` when a window date has no genuine observation (the value was forward-filled, or the date is absent from the data, e.g. a weekend).

In [78]:
import sys
import json
import holidays
import pandas as pd
import numpy as np
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

sys.path.append('..')
from modules.source import START, END, Sources, parquet_daily, parquet_fomc, OUTPUT_FOLDER_PATH

with open('../modules/fomc_dates.json') as f:
   fomc_meta = json.load(f)
fomc_dates = pd.to_datetime(fomc_meta['dates']) + pd.Timedelta(days=1)
offset = (-1, 0, 1)
fomc_window = pd.DatetimeIndex([date + pd.Timedelta(days=i) for date in fomc_dates for i in offset])

df_main = pd.read_parquet(parquet_daily)
weekdays = pd.bdate_range(df_main.index.min(), df_main.index.max())
my_holidays = pd.DatetimeIndex(holidays.Malaysia(subdiv='KUL', years=range(START.year, END.year + 1)).keys())

asset_names = [Sources.MGS_10Y, Sources.KLCI, Sources.USDMYR]
df_bool = df_main.xs('is_ffilled', axis=1, level=1)[asset_names].reindex(fomc_window, fill_value=True) # True = data mising

### 2. Diagnose missing data in the event window

Every flagged window date is attributed to one cause:

1. **Weekend:** the date is not a weekday.
2. **Malaysian public holiday:** a weekday on the Kuala Lumpur holiday calendar.
3. **Unexplained:** any remaining gap. This points to a data problem rather than a market closure.

**Result:** across 94 FOMC events (282 event-window sessions), the only unexplained gaps are 8 MGS 10Y observations spread over 7 events. KLCI and USDMYR have none. These 7 events are excluded. USDMYR also shows no holiday gaps, unlike MGS 10Y and KLCI (24 each), so it is still quoted on Malaysian holidays.

In [79]:
is_weekend = ~df_bool.index.isin(weekdays)
is_holiday = df_bool.index.isin(my_holidays) & ~is_weekend # exclude weekend from holidays
is_unexplained = df_bool.any(axis=1) & ~is_weekend & ~is_holiday # remaining forward filled data are unexplained

summary_df = pd.DataFrame(
   [df_bool.sum(), df_bool[is_weekend].sum(), df_bool[is_holiday].sum(), df_bool[is_unexplained].sum()], 
   index=['Missing Days', ' - Explained by Weekends', ' - Explained by Holidays', '= Unexplained']
)

m_index = pd.MultiIndex.from_product([fomc_dates, ['t-1', 't+0', 't+1']], names=['fomc_date', 'offset'])
df_unexplained = df_bool[is_unexplained].set_axis(m_index[is_unexplained])
affected_fomc_dates = df_unexplained.index.get_level_values('fomc_date')

print(
   "\nMissing data attribution:\n"
   f"{summary_df}\n"
   " (Expanded below)\n"
   f"{df_unexplained.astype(object).replace({True: '1', False: '-'}).to_string(col_space=8)}\n"
   '-----------------------------\n'
   f"Total FOMC events evaluated: {len(fomc_dates)} ({len(fomc_window)} event-window sessions)\n"
   f"Excluded due to unexplained data gaps: {affected_fomc_dates.nunique()} events"
)


Missing data attribution:
                          MGS_10Y  KLCI  USDMYR
Missing Days                   37    29       5
 - Explained by Weekends        5     5       5
 - Explained by Holidays       24    24       0
= Unexplained                   8     0       0
 (Expanded below)
                     MGS_10Y     KLCI   USDMYR
fomc_date  offset                             
2016-07-28 t-1             1        -        -
2017-06-15 t-1             1        -        -
           t+1             1        -        -
2017-12-14 t+1             1        -        -
2018-12-20 t+0             1        -        -
2019-03-21 t-1             1        -        -
2021-11-04 t-1             1        -        -
2022-11-03 t+1             1        -        -
-----------------------------
Total FOMC events evaluated: 94 (282 event-window sessions)
Excluded due to unexplained data gaps: 7 events


### 3. Define event dates and classify events

Weekend and holiday gaps are not data errors, but they change what a "1-day" window means. So instead of fixed calendar offsets, the window is built from days on which all three assets jointly have a genuine (non-forward-filled) observation:

- `T_post0` is the first joint trading day on or after the event date. `T_pre` and `T_post1` are the joint trading days immediately before and after it.
- `calendar_gap_days` is `T_post1` minus `T_pre`. A normal window spans 2 calendar days.

Each event receives a status:

| Status | Rule |
|---|---|
| `VALID` | No unexplained gap and `calendar_gap_days` ≤ 4 |
| `DROP_EXTENDED_HOLIDAY` | `calendar_gap_days` > 4, so a multi-day closure stretches the window beyond one day either side |
| `DROP_INVALID_DATA` | Unexplained data gap from section 2. Window dates are set to `NA`. This takes priority over the holiday rule |

**Result:** 72 valid events, 15 dropped for extended holidays and 7 dropped for invalid data (94 in total). Most windows span exactly 2 calendar days (61 events), and the 15 extended-holiday drops are the events with gaps of 5 to 7 days.

In [80]:
price_change = df_main.xs('change', axis=1, level=1)[asset_names]
ffilled = df_main.xs('is_ffilled', axis=1, level=1)[asset_names]
df_pchg_noffill = price_change.where(~ffilled).dropna(how='any')

idx_post0 = df_pchg_noffill.index.searchsorted(fomc_dates)
t_pre, t_post0, t_post1 = (df_pchg_noffill.index[idx_post0 + i] for i in offset)
calendar_gap = (t_post1 - t_pre).days

status = np.select(
   [fomc_dates.isin(affected_fomc_dates), calendar_gap > 4], 
   ['DROP_INVALID_DATA', 'DROP_EXTENDED_HOLIDAY'], 
   default='VALID'
)

# Instantiate Int64 directly inside constructor
fomc_dataset = pd.DataFrame({'fomc_date': fomc_dates.date, 'T_pre': t_pre.date, 'T_post0': t_post0.date, 'T_post1': t_post1.date, 'calendar_gap_days': pd.Series(calendar_gap, dtype='Int64'), 'status': status}) # cast dtype so pd.NA wont force int into float
fomc_dataset.loc[fomc_dataset['status'] == 'DROP_INVALID_DATA', ['T_pre', 'T_post0', 'T_post1', 'calendar_gap_days']] = pd.NA

print(
   "\n=== FOMC EVENT WINDOW DIAGNOSTIC SUMMARY ===\n"
   f"{fomc_dataset['status'].value_counts().to_string()}\n\n"
   "=== Calendar Gap Frequency Breakdown ===\n"
   f"{fomc_dataset['calendar_gap_days'].value_counts().sort_index().to_frame('count')}\n\n"
   "Non-Valid Events Filtered Out:\n\n"
   f"{fomc_dataset[fomc_dataset['status'] != 'VALID'].to_string(index=False, col_space=12)}\n\n"
)


=== FOMC EVENT WINDOW DIAGNOSTIC SUMMARY ===
status
VALID                    72
DROP_EXTENDED_HOLIDAY    15
DROP_INVALID_DATA         7

=== Calendar Gap Frequency Breakdown ===
                   count
calendar_gap_days       
2                     61
3                      6
4                      5
5                      8
6                      6
7                      1

Non-Valid Events Filtered Out:

   fomc_date        T_pre      T_post0      T_post1  calendar_gap_days                status
  2015-04-30   2015-04-29   2015-04-30   2015-05-05                  6 DROP_EXTENDED_HOLIDAY
  2016-07-28         <NA>         <NA>         <NA>               <NA>     DROP_INVALID_DATA
  2017-06-15         <NA>         <NA>         <NA>               <NA>     DROP_INVALID_DATA
  2017-09-21   2017-09-20   2017-09-21   2017-09-25                  5 DROP_EXTENDED_HOLIDAY
  2017-12-14         <NA>         <NA>         <NA>               <NA>     DROP_INVALID_DATA
  2018-02-01   2018-01-30   20

### 4. Build the event-window price dataset

Keep only `VALID` events and reshape them to long format (three row per event and window date: `T_pre`, `T_post0`, `T_post1`). Then attach the MGS 10Y, KLCI and USDMYR values for each date.

The result has 216 rows (72 events × 3 dates) and is saved to `parquet_fomc` for the downstream analysis.

In [81]:
WINDOW_COLS = ['T_pre', 'T_post0', 'T_post1']

long_dates = fomc_dataset[fomc_dataset['status'] == 'VALID'].melt('fomc_date', WINDOW_COLS, 'window', 'date')
long_dates['window'] = pd.Categorical(long_dates['window'], categories=WINDOW_COLS, ordered=True)
long_dates['date'] = pd.to_datetime(long_dates['date'])

df_event_pchg = (
   long_dates.join(df_pchg_noffill, on='date')
   .sort_values(['fomc_date', 'window'])
   .reset_index(drop=True)
)

df_event_pchg.to_parquet(parquet_fomc)
print(df_event_pchg)

      fomc_date   window       date  MGS_10Y      KLCI    USDMYR
0    2015-01-29    T_pre 2015-01-28    -0.06 -0.004051  0.000000
1    2015-01-29  T_post0 2015-01-29     0.01 -0.007658  0.000416
2    2015-01-29  T_post1 2015-01-30    -0.04 -0.000516  0.007467
3    2015-03-19    T_pre 2015-03-18     0.02  0.005411  0.001732
4    2015-03-19  T_post0 2015-03-19    -0.03  0.006410 -0.013912
..          ...      ...        ...      ...       ...       ...
211  2026-06-18  T_post0 2026-06-18    -0.01  0.000818  0.004609
212  2026-06-18  T_post1 2026-06-19     0.01  0.000374  0.002726
213  2026-07-30    T_pre 2026-07-29     0.02  0.001797 -0.001199
214  2026-07-30  T_post0 2026-07-30     0.00  0.002817  0.001101
215  2026-07-30  T_post1 2026-07-31    -0.02  0.002612 -0.000294

[216 rows x 6 columns]


### 5. Export event diagnostics to Excel

Write the full event table (`fomc_dataset`, all 94 events including dropped ones) to `fomc_event_diagnostics.xlsx` for manual review. Column widths are auto-fitted and each row is colour-coded by status: green for `VALID`, yellow for `DROP_EXTENDED_HOLIDAY`, red for `DROP_INVALID_DATA`.

In [82]:
STATUS_FILL = {
   'VALID':                 'C6EFCE',  # green
   'DROP_EXTENDED_HOLIDAY': 'FFEB9C',  # yellow
   'DROP_INVALID_DATA':     'FFC7CE',  # red
}

output_path = f"{OUTPUT_FOLDER_PATH}/fomc_event_diagnostics.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
   fomc_dataset.to_excel(writer, index=False)
   ws = writer.book.active

   shown = fomc_dataset.astype(str)
   for i, col in enumerate(fomc_dataset.columns, start=1):
      ws.column_dimensions[get_column_letter(i)].width = max(shown[col].str.len().max(), len(col)) + 3

   for row, status in zip(ws.iter_rows(min_row=2), fomc_dataset['status']):  # row 1 = header
      for cell in row:
         cell.fill = PatternFill('solid', fgColor=STATUS_FILL[status])

print(f"Saved: {output_path}")

Saved: ../output/fomc_event_diagnostics.xlsx
